In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")

catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "210_build_silver_closed_bids_base_clean.py"
RUN_ID = str(uuid.uuid4())

SOURCE_TABLE = f"{catalog}.bronze.closed_bids_raw"
TARGET_TABLE = f"{catalog}.silver.closed_bids_base_clean"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build silver.closed_bids_base_clean
# ============================================================
try:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE}
    USING DELTA
    AS
    WITH src AS (
        SELECT
              socrata_id                                                       AS source_row_id
            , CAST('closed_bids' AS STRING)                                    AS source_name
            , NULLIF(TRIM(socrata_created_at), '')                             AS source_created_at_raw
            , NULLIF(TRIM(socrata_updated_at), '')                             AS source_updated_at_raw
            , NULLIF(TRIM(socrata_version), '')                                AS source_version

            , NULLIF(TRIM(ingestion_run_id), '')                               AS ingestion_run_id
            , NULLIF(TRIM(ingested_at_utc), '')                                AS ingested_at_utc_raw

            , NULLIF(TRIM(project_id), '')                                     AS project_id
            , NULLIF(TRIM(project_name), '')                                   AS project_name
            , NULLIF(TRIM(project_number), '')                                 AS project_number
            , NULLIF(TRIM(state_project_number), '')                           AS state_project_number
            , NULLIF(TRIM(control_section_job_csj), '')                        AS control_section_job_csj
            , NULLIF(TRIM(controlling_project_id_ccsj), '')                    AS controlling_project_id_ccsj
            , NULLIF(TRIM(project_type), '')                                   AS project_type
            , NULLIF(TRIM(project_classification), '')                         AS project_classification
            , NULLIF(TRIM(project_actual_let_date), '')                        AS project_actual_let_date_raw
            , NULLIF(TRIM(project_estimated_let_date), '')                     AS project_estimated_let_date_raw
            , NULLIF(TRIM(project_limits_from), '')                            AS project_limits_from
            , NULLIF(TRIM(project_limits_to), '')                              AS project_limits_to
            , NULLIF(TRIM(county), '')                                         AS county
            , NULLIF(TRIM(district_division), '')                              AS district_division
            , NULLIF(TRIM(highway), '')                                        AS highway
            , NULLIF(TRIM(short_description), '')                              AS short_description
            , NULLIF(TRIM(federal_project_number), '')                         AS federal_project_number

            , NULLIF(TRIM(vendor_name), '')                                    AS vendor_name_raw
            , NULLIF(TRIM(bid_submit_sequence_number), '')                     AS bid_submit_sequence_number_raw
            , NULLIF(TRIM(bid_rank_sequence_number), '')                       AS bid_rank_sequence_number_raw
            , NULLIF(TRIM(low_bidder_flag), '')                                AS low_bidder_flag_raw
            , NULLIF(TRIM(dbe_goal_percent), '')                               AS dbe_goal_percent_raw
            , NULLIF(TRIM(maximum_number_of_working), '')                      AS maximum_number_of_working_raw

            , NULLIF(TRIM(bid_code), '')                                       AS bid_code
            , NULLIF(TRIM(alternative_bid_code), '')                           AS alternative_bid_code
            , NULLIF(TRIM(bid_item_sequence_number), '')                       AS bid_item_sequence_number_raw
            , NULLIF(TRIM(bid_item_description), '')                           AS bid_item_description
            , NULLIF(TRIM(measurement_unit), '')                               AS measurement_unit
            , NULLIF(TRIM(bid_item_quantity), '')                              AS bid_item_quantity_raw
            , NULLIF(TRIM(bid_item_unit_price_amount), '')                     AS bid_item_unit_price_amount_raw
            , NULLIF(TRIM(bid_total_amount), '')                               AS bid_total_amount_raw

            , NULLIF(TRIM(specification_code), '')                             AS specification_code_raw
            , NULLIF(TRIM(specification_description), '')                      AS specification_description
            , NULLIF(TRIM(spec_book_year), '')                                 AS spec_book_year_raw

            , NULLIF(TRIM(engineer_s_estimate_unit), '')                       AS engineer_s_estimate_unit_raw
            , NULLIF(TRIM(sealed_engineer_s_estimate), '')                     AS sealed_engineer_s_estimate_raw
            , NULLIF(TRIM(sealed_engineer_s_estimate_1), '')                   AS sealed_engineer_s_estimate_1_raw
            , NULLIF(TRIM(a_b_engineer_s_estimate_amount), '')                 AS a_b_engineer_s_estimate_amount_raw

            , NULLIF(TRIM(force_account_amount), '')                           AS force_account_amount_raw
            , NULLIF(TRIM(force_account_description), '')                      AS force_account_description
            , NULLIF(TRIM(nbi_number), '')                                     AS nbi_number
            , NULLIF(TRIM(utility_id), '')                                     AS utility_id
        FROM {SOURCE_TABLE}
    )

    , typed AS (
        SELECT
              source_name
            , source_row_id

            , CASE
                  WHEN source_created_at_raw RLIKE '^[0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}}T'
                  THEN CAST(source_created_at_raw AS TIMESTAMP)
                  ELSE NULL
              END                                                               AS source_created_at

            , CASE
                  WHEN source_updated_at_raw RLIKE '^[0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}}T'
                  THEN CAST(source_updated_at_raw AS TIMESTAMP)
                  ELSE NULL
              END                                                               AS source_updated_at

            , source_version
            , ingestion_run_id

            , CASE
                  WHEN ingested_at_utc_raw RLIKE '^[0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}}T'
                  THEN CAST(ingested_at_utc_raw AS TIMESTAMP)
                  ELSE NULL
              END                                                               AS ingested_at_utc

            , project_id
            , project_name
            , project_number
            , state_project_number
            , control_section_job_csj
            , controlling_project_id_ccsj
            , project_type
            , project_classification

            , CASE
                  WHEN project_actual_let_date_raw RLIKE '^[0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}}'
                  THEN CAST(project_actual_let_date_raw AS DATE)
                  ELSE NULL
              END                                                               AS project_actual_let_date

            , CASE
                  WHEN project_estimated_let_date_raw RLIKE '^[0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}}'
                  THEN CAST(project_estimated_let_date_raw AS DATE)
                  ELSE NULL
              END                                                               AS project_estimated_let_date

            , project_limits_from
            , project_limits_to
            , county
            , district_division
            , highway
            , short_description
            , federal_project_number

            , vendor_name_raw                                                   AS vendor_name
            , LOWER(TRIM(COALESCE(vendor_name_raw, '')))                        AS vendor_name_lower
            , UPPER(TRIM(COALESCE(vendor_name_raw, '')))                        AS vendor_name_standardized

            , CASE
                  WHEN bid_submit_sequence_number_raw RLIKE '^-?[0-9]+(\\.[0-9]+)?$'
                  THEN CAST(CAST(bid_submit_sequence_number_raw AS DECIMAL(38,10)) AS BIGINT)
                  ELSE NULL
              END                                                               AS bid_submit_sequence_number

            , CASE
                  WHEN bid_rank_sequence_number_raw RLIKE '^-?[0-9]+(\\.[0-9]+)?$'
                  THEN CAST(CAST(bid_rank_sequence_number_raw AS DECIMAL(38,10)) AS BIGINT)
                  ELSE NULL
              END                                                               AS bid_rank_sequence_number

            , CASE
                  WHEN LOWER(COALESCE(low_bidder_flag_raw, '')) IN ('true', 't', '1', 'yes', 'y') THEN TRUE
                  WHEN LOWER(COALESCE(low_bidder_flag_raw, '')) IN ('false', 'f', '0', 'no', 'n') THEN FALSE
                  ELSE NULL
              END                                                               AS low_bidder_flag

            , CASE
                  WHEN dbe_goal_percent_raw RLIKE '^-?[0-9]+(\\.[0-9]+)?$'
                  THEN CAST(dbe_goal_percent_raw AS DECIMAL(38,10))
                  ELSE NULL
              END                                                               AS dbe_goal_percent

            , CASE
                  WHEN maximum_number_of_working_raw RLIKE '^-?[0-9]+(\\.[0-9]+)?$'
                  THEN CAST(maximum_number_of_working_raw AS DECIMAL(38,10))
                  ELSE NULL
              END                                                               AS maximum_number_of_working

            , bid_code
            , alternative_bid_code

            , CASE
                  WHEN bid_item_sequence_number_raw RLIKE '^-?[0-9]+(\\.[0-9]+)?$'
                  THEN CAST(CAST(bid_item_sequence_number_raw AS DECIMAL(38,10)) AS BIGINT)
                  ELSE NULL
              END                                                               AS bid_item_sequence_number

            , bid_item_description
            , measurement_unit

            , CASE
                  WHEN bid_item_quantity_raw RLIKE '^-?[0-9]+(\\.[0-9]+)?$'
                  THEN CAST(bid_item_quantity_raw AS DECIMAL(38,10))
                  ELSE NULL
              END                                                               AS bid_item_quantity

            , CASE
                  WHEN bid_item_unit_price_amount_raw RLIKE '^-?[0-9]+(\\.[0-9]+)?$'
                  THEN CAST(bid_item_unit_price_amount_raw AS DECIMAL(38,10))
                  ELSE NULL
              END                                                               AS bid_item_unit_price_amount

            , CASE
                  WHEN bid_total_amount_raw RLIKE '^-?[0-9]+(\\.[0-9]+)?$'
                  THEN CAST(bid_total_amount_raw AS DECIMAL(38,10))
                  ELSE NULL
              END                                                               AS bid_total_amount

            , CASE
                  WHEN specification_code_raw RLIKE '^-?[0-9]+(\\.[0-9]+)?$'
                  THEN CAST(specification_code_raw AS DECIMAL(38,10))
                  ELSE NULL
              END                                                               AS specification_code

            , specification_description

            , CASE
                  WHEN spec_book_year_raw RLIKE '^-?[0-9]+$'
                  THEN CAST(spec_book_year_raw AS INT)
                  ELSE NULL
              END                                                               AS spec_book_year

            , CASE
                  WHEN engineer_s_estimate_unit_raw RLIKE '^-?[0-9]+(\\.[0-9]+)?$'
                  THEN CAST(engineer_s_estimate_unit_raw AS DECIMAL(38,10))
                  ELSE NULL
              END                                                               AS engineer_s_estimate_unit

            , CASE
                  WHEN sealed_engineer_s_estimate_raw RLIKE '^-?[0-9]+(\\.[0-9]+)?$'
                  THEN CAST(sealed_engineer_s_estimate_raw AS DECIMAL(38,10))
                  ELSE NULL
              END                                                               AS sealed_engineer_s_estimate

            , CASE
                  WHEN sealed_engineer_s_estimate_1_raw RLIKE '^-?[0-9]+(\\.[0-9]+)?$'
                  THEN CAST(sealed_engineer_s_estimate_1_raw AS DECIMAL(38,10))
                  ELSE NULL
              END                                                               AS sealed_engineer_s_estimate_1

            , CASE
                  WHEN a_b_engineer_s_estimate_amount_raw RLIKE '^-?[0-9]+(\\.[0-9]+)?$'
                  THEN CAST(a_b_engineer_s_estimate_amount_raw AS DECIMAL(38,10))
                  ELSE NULL
              END                                                               AS a_b_engineer_s_estimate_amount

            , CASE
                  WHEN force_account_amount_raw RLIKE '^-?[0-9]+(\\.[0-9]+)?$'
                  THEN CAST(force_account_amount_raw AS DECIMAL(38,10))
                  ELSE NULL
              END                                                               AS force_account_amount

            , force_account_description
            , nbi_number
            , utility_id
        FROM src
    )

    , derived AS (
        SELECT
              *

            , CASE
                  WHEN vendor_name_lower LIKE 'txdot%'
                  THEN TRUE
                  ELSE FALSE
              END                                                               AS is_engineer_estimate_row

            , CASE
                  WHEN vendor_name_lower LIKE 'txdot%'
                  THEN 'ENGINEER_ESTIMATE'
                  ELSE 'VENDOR'
              END                                                               AS row_type

            , COALESCE(
                  a_b_engineer_s_estimate_amount
                , sealed_engineer_s_estimate
                , sealed_engineer_s_estimate_1
              )                                                                 AS engineer_project_total

            , engineer_s_estimate_unit                                          AS engineer_estimate_unit_price

            , (bid_item_quantity IS NOT NULL)                                   AS is_valid_quantity
            , (bid_item_unit_price_amount IS NOT NULL)                          AS is_valid_unit_price
            , (bid_total_amount IS NOT NULL)                                    AS is_valid_bid_total

            , CONCAT_WS(
                  '|'
                , COALESCE(project_id, '')
                , COALESCE(vendor_name, '')
                , COALESCE(CAST(bid_submit_sequence_number AS STRING), '')
              )                                                                 AS business_submission_key

            , CONCAT_WS(
                  '|'
                , COALESCE(project_id, '')
                , COALESCE(vendor_name, '')
                , COALESCE(CAST(bid_submit_sequence_number AS STRING), '')
                , COALESCE(CAST(bid_item_sequence_number AS STRING), '')
              )                                                                 AS business_item_key

            , md5(CONCAT_WS(
                  '|'
                , COALESCE(source_row_id, '')
                , COALESCE(project_id, '')
                , COALESCE(vendor_name, '')
                , COALESCE(CAST(bid_submit_sequence_number AS STRING), '')
                , COALESCE(CAST(bid_item_sequence_number AS STRING), '')
                , COALESCE(bid_code, '')
                , COALESCE(CAST(specification_code AS STRING), '')
                , COALESCE(CAST(bid_item_quantity AS STRING), '')
                , COALESCE(CAST(bid_item_unit_price_amount AS STRING), '')
                , COALESCE(CAST(bid_total_amount AS STRING), '')
                , COALESCE(CAST(source_updated_at AS STRING), '')
              ))                                                                AS record_hash
        FROM typed
    )

    , ranked AS (
        SELECT
              *
            , COALESCE(source_row_id, record_hash)                              AS base_row_key
            , ROW_NUMBER() OVER (
                PARTITION BY COALESCE(source_row_id, record_hash)
                ORDER BY
                      source_updated_at DESC NULLS LAST
                    , ingested_at_utc DESC NULLS LAST
                    , source_row_id DESC NULLS LAST
              )                                                                 AS dedupe_rank
        FROM derived
    )

    , final_with_diag AS (
        SELECT
              *
            , (COUNT(*) OVER (PARTITION BY business_item_key) > 1)              AS is_duplicate_business_item_key
        FROM ranked
    )

    SELECT
          source_name
        , base_row_key
        , source_row_id
        , source_created_at
        , source_updated_at
        , source_version
        , ingestion_run_id
        , ingested_at_utc

        , project_id
        , project_name
        , project_number
        , state_project_number
        , control_section_job_csj
        , controlling_project_id_ccsj
        , project_type
        , project_classification
        , project_actual_let_date
        , project_estimated_let_date
        , project_limits_from
        , project_limits_to
        , county
        , district_division
        , highway
        , short_description
        , federal_project_number

        , vendor_name
        , vendor_name_standardized
        , is_engineer_estimate_row
        , bid_submit_sequence_number
        , bid_rank_sequence_number
        , low_bidder_flag
        , dbe_goal_percent
        , maximum_number_of_working

        , bid_code
        , alternative_bid_code
        , bid_item_sequence_number
        , bid_item_description
        , measurement_unit
        , bid_item_quantity
        , bid_item_unit_price_amount
        , bid_total_amount

        , specification_code
        , specification_description
        , spec_book_year

        , engineer_s_estimate_unit
        , sealed_engineer_s_estimate
        , sealed_engineer_s_estimate_1
        , a_b_engineer_s_estimate_amount
        , engineer_estimate_unit_price
        , engineer_project_total

        , force_account_amount
        , force_account_description
        , nbi_number
        , utility_id

        , row_type
        , is_valid_quantity
        , is_valid_unit_price
        , is_valid_bid_total

        , business_submission_key
        , business_item_key
        , is_duplicate_business_item_key
        , record_hash
        , TRUE                                                                  AS is_current_row
        , dedupe_rank
    FROM final_with_diag
    WHERE dedupe_rank = 1
    """)

    spark.sql(f"""
    COMMENT ON TABLE {TARGET_TABLE} IS
    'Core cleaned silver table built from bronze.closed_bids_raw. Applies trimming, typing, standardization, derived business fields, and deduplication logic.'
    """)

    row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {TARGET_TABLE}").collect()[0]["row_count"]

    log_run(
        "SUCCESS",
        row_count,
        f"Built {TARGET_TABLE} successfully."
    )

    print("=" * 70)
    print(f"Built {TARGET_TABLE}")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise